In [8]:
import tensorflow as tf
import keras
import numpy as np
import pandas as pd


In [10]:
import os
dataset_dir = r"C:\Users\NEW\Downloads\dog-breed-identification\train"
print("Total images:", len(os.listdir(dataset_dir)))


Total images: 10222


In [11]:
import pandas as pd

labels = pd.read_csv(r"C:\Users\NEW\Downloads\dog-breed-identification\labels.csv")
labels.head()


,id,breed
0,000bec180eb18c7604dcecc8fe0dba07,boston_bull
1,001513dfcb2ffafc82cccf4d8bbaba97,dingo
2,001cdf01b096e06d78e9e5112d419397,pekinese
3,00214f311d5d2247d5dfe4fe24b2303d,bluetick
4,0021f9ceb3235effd7fcde7f7538ed62,golden_retriever


In [14]:
import os
import shutil
import pandas as pd


In [16]:
dataset_dir = r"C:\Users\NEW\Downloads\dog-breed-identification\train"
labels = pd.read_csv(r"C:\Users\NEW\Downloads\dog-breed-identification\labels.csv")


In [18]:
base_dir = r"C:\Users\NEW\Downloads\dog-breed-identification\subset"

if not os.path.exists(base_dir):
    os.makedirs(base_dir)

train_dir = os.path.join(base_dir, 'train')

if not os.path.exists(train_dir):
    os.makedirs(train_dir)


In [20]:
breeds = labels['breed'].unique()

for breed in breeds:
    breed_folder = os.path.join(train_dir, breed)
    if not os.path.exists(breed_folder):
        os.makedirs(breed_folder)


In [22]:
import shutil

for index, row in labels.iterrows():
    img_name = row['id'] + ".jpg"
    breed = row['breed']
    
    source = os.path.join(dataset_dir, img_name)
    destination = os.path.join(train_dir, breed, img_name)
    
    if os.path.exists(source):
        shutil.copy(source, destination)


In [24]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [26]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)


In [28]:
train_generator = train_datagen.flow_from_directory(
    r"C:\Users\NEW\Downloads\dog-breed-identification\subset\train",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)


Found 8221 images belonging to 120 classes.


In [30]:
validation_generator = train_datagen.flow_from_directory(
    r"C:\Users\NEW\Downloads\dog-breed-identification\subset\train",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)


Found 2001 images belonging to 120 classes.


In [32]:
from tensorflow.keras.applications import VGG19
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.models import Model


In [34]:
IMAGE_SIZE = [224, 224]

vgg = VGG19(
    input_shape=IMAGE_SIZE + [3],
    weights='imagenet',
    include_top=False
)


In [35]:
for layer in vgg.layers:
    layer.trainable = False


In [38]:
# Flatten the output of VGG19
x = Flatten()(vgg.output)

# Output layer (120 dog breeds)
prediction = Dense(120, activation='softmax')(x)

# Final model
model = Model(inputs=vgg.input, outputs=prediction)


In [40]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


In [42]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=6
)

C:\Users\NEW\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/6
257/257 ━━━━━━━━━━━━━━━━━━━━ 2556s 10s/step - accuracy: 0.0353 - loss: 6.9610 - val_accuracy: 0.0750 - val_loss: 5.7763
Epoch 2/6
257/257 ━━━━━━━━━━━━━━━━━━━━ 2823s 11s/step - accuracy: 0.1214 - loss: 5.3916 - val_accuracy: 0.0975 - val_loss: 6.0228
Epoch 3/6
257/257 ━━━━━━━━━━━━━━━━━━━━ 2517s 10s/step - accuracy: 0.1767 - loss: 4.9177 - val_accuracy: 0.1129 - val_loss: 5.9971
Epoch 4/6
257/257 ━━━━━━━━━━━━━━━━━━━━ 2626s 10s/step - accuracy: 0.2095 - loss: 4.7000 - val_accuracy: 0.0975 - val_loss: 6.2679
Epoch 5/6
257/257 ━━━━━━━━━━━━━━━━━━━━ 2433s 9s/step - accuracy: 0.2281 - loss: 4.5032 - val_accuracy: 0.1279 - val_loss: 6.0743
Epoch 6/6
257/257 ━━━━━━━━━━━━━━━━━━━━ 2626s 10s/step - accuracy: 0.2645 - loss: 4.3168 - val_accuracy: 0.1284 - val_loss: 6.3589


In [44]:
model.save("dogbreed.h5")

In [4]:
import numpy as np
from tensorflow.keras.preprocessing import image

In [40]:
class_indices = train_generator.class_indices

# Convert dictionary to list
class_names = list(class_indices.keys())

print(class_names[:10])

['affenpinscher', 'afghan_hound', 'african_hunting_dog', 'airedale', 'american_staffordshire_terrier', 'appenzeller', 'australian_terrier', 'basenji', 'basset', 'beagle']


In [42]:
img_path = r"C:\Users\NEW\Downloads\dog-breed-identification\train\000bec180eb18c7604dcecc8fe0dba07.jpg"

from tensorflow.keras.preprocessing import image
import numpy as np

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

pred = model.predict(img_array)
predicted_index = np.argmax(pred)

print("Predicted breed:", class_names[predicted_index])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Predicted breed: border_terrier


In [51]:
import tensorflow as tf
tf.config.run_functions_eagerly(True)

In [53]:
from tensorflow.keras.models import load_model
model = load_model("dogbreed.h5")

In [55]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
model.fit(train_generator, validation_data=validation_generator, epochs=6)

Epoch 1/6


C:\Users\NEW\anaconda3\Lib\site-packages\tensorflow\python\data\ops\structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


 87/257 ━━━━━━━━━━━━━━━━━━━━ 1:29:55 32s/step - accuracy: 0.2492 - loss: 5.3138